# Блок 4. Построение ML моделей.

## Описание блока:
В данном блоке будут обучены ML модели с применением кросс валидации и подбора лучших гиперпараметров. Подробное текстовое описания получившегося результата будет в  [Открыть тетрадку "Part 0"](Part%200.%20Research%20results.ipynb)

In [1]:
# импорт основных библиотек блока
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from functools import partial
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMRegressor

import sys
import os

# Добавляем родительскую директорию в sys.path
sys.path.append(os.path.abspath(".."))

from all_functions.datasets_load import load_all_datasets
from all_functions.datasets_transfom import *
from all_functions.func_for_ml import * 

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# фиксация сида на будущее
RANDOM_STATE = 654321

In [3]:
# загрузка датасетов
crypt_Ethereum, crypt_BTC, futures_Brent_LCOK5, \
futures_WTI_CLJ5, share_metal_jiangxi_copper, \
share_metal_baoshan_iron, share_oil_petrochina_hk, \
share_oil_equinor_oslo, spot_preciouse_metal_AG, spot_preciouse_metal_AU = load_all_datasets()

In [4]:
# трансформация датасетов
crypt_Ethereum = transform_data(crypt_Ethereum)
crypt_BTC = transform_data(crypt_BTC)

futures_WTI_CLJ5 = transform_data(futures_WTI_CLJ5)
futures_Brent_LCOK5 = transform_data(futures_Brent_LCOK5)

share_metal_jiangxi_copper  = transform_data(share_metal_jiangxi_copper)
share_metal_baoshan_iron  = transform_data(share_metal_baoshan_iron)

share_oil_petrochina_hk  = transform_data(share_oil_petrochina_hk)
share_oil_equinor_oslo  = transform_data(share_oil_equinor_oslo)

spot_preciouse_metal_AU  = transform_data(spot_preciouse_metal_AU)
spot_preciouse_metal_AG = transform_data(spot_preciouse_metal_AG)

## Построение ML моделей

In [5]:
all_data = {
    'crypt_Ethereum': crypt_Ethereum, 
    'crypt_BTC':crypt_BTC,
    'futures_Brent_LCOK5':futures_Brent_LCOK5,
    'futures_WTI_CLJ5': futures_WTI_CLJ5,
    'share_metal_jiangxi_copper': share_metal_jiangxi_copper, 
    'share_metal_baoshan_iron': share_metal_baoshan_iron,
    'share_oil_petrochina_hk': share_oil_petrochina_hk, 
    'share_oil_equinor_oslo': share_oil_equinor_oslo,
    'spot_preciouse_metal_AG': spot_preciouse_metal_AG,
    'spot_preciouse_metal_AU': spot_preciouse_metal_AU
}


In [6]:
# Создание датасета для записи результатов обучения
all_reault = pd.DataFrame()

name_ds = []
for i, _ in all_data.items():
    name_ds.append(i)
all_reault['dataset_name'] = name_ds

# фиксация в общем датасете средней цены, для наглядности диапазона цен

mean_ds = []
for i, n in all_data.items():
    mean_ds.append(n.mean()['Цена'])
all_reault['dataset_mean_price'] = mean_ds

all_reault

,dataset_name,dataset_mean_price
0,crypt_Ethereum,1264.224518
1,crypt_BTC,24007.290015
2,futures_Brent_LCOK5,68.036667
3,futures_WTI_CLJ5,63.524965
4,share_metal_jiangxi_copper,10.239784
5,share_metal_baoshan_iron,5.255690
6,share_oil_petrochina_hk,3.622050
7,share_oil_equinor_oslo,187.087574
8,spot_preciouse_metal_AG,20.814301
9,spot_preciouse_metal_AU,1677.948521


In [7]:
def objective_RFregresson(trial, data):
    
    # --- Гиперпараметры для Random Forest ---
    n_estimators = trial.suggest_int("n_estimators", 50, 300)
    max_depth = trial.suggest_int("max_depth", 3, 20)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 5)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2"])

    # --- Разделение данных ---
    train_raw, valid_raw, _ = split_data(data)
    tail_len = 30

    train = upgrade_dataset(train_raw)
    valid_with_tail = pd.concat([train_raw.tail(tail_len), valid_raw])
    valid_with_tail = upgrade_dataset(valid_with_tail)
    valid = valid_with_tail.loc[valid_raw.index]

    X_train, y_train = return_x_y(train)
    X_valid, y_valid = return_x_y(valid)

    # --- Модель ---
    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_valid)

    rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
    
    mape = mean_absolute_percentage_error(y_valid, y_pred)
    trial.set_user_attr("mape", mape)
    
    return rmse

In [8]:
%%time

result_best_RF_rmse = []
result_best_RF_mape = []

for i, n in all_data.items():

    line_length = 150
    print('-' * line_length)
    print(f'Лучшие параметры для датасета (модель RandomForestRegressor): {i}'.center(line_length))
    print('-' * line_length)

    optuna.logging.set_verbosity(optuna.logging.ERROR)
    study = optuna.create_study(direction="minimize")
    study.optimize(partial(objective_RFregresson, data=n), n_trials=10)

    print("Лучшие параметры:", study.best_params)
    print("Лучшее RMSE:", study.best_value)

    result_best_RF_rmse.append(study.best_value)

    df = study.trials_dataframe()
    best_row = df.loc[df["value"].idxmin()]
    best_mape = round(best_row["user_attrs_mape"] * 100, 4)
    result_best_RF_mape.append(best_mape)

all_reault['RF_result_rmse'] = result_best_RF_rmse
all_reault['RF_result_mape'] = result_best_RF_mape

------------------------------------------------------------------------------------------------------------------------------------------------------
                                     Лучшие параметры для датасета (модель RandomForestRegressor): crypt_Ethereum                                     
------------------------------------------------------------------------------------------------------------------------------------------------------
Лучшие параметры: {'n_estimators': 103, 'max_depth': 13, 'min_samples_split': 8, 'min_samples_leaf': 4, 'max_features': 'sqrt'}
Лучшее RMSE: 125.91967789922586
------------------------------------------------------------------------------------------------------------------------------------------------------
                                       Лучшие параметры для датасета (модель RandomForestRegressor): crypt_BTC                                        
-------------------------------------------------------------------------------------

In [9]:
def objective_gbr(trial, data):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5)
    }

    train_raw, valid_raw, _ = split_data(data)
    tail_len = 30

    train = upgrade_dataset(train_raw)
    valid_with_tail = pd.concat([train_raw.tail(tail_len), valid_raw])
    valid_with_tail = upgrade_dataset(valid_with_tail)
    valid = valid_with_tail.loc[valid_raw.index]

    X_train, y_train = return_x_y(train)
    X_valid, y_valid = return_x_y(valid)

    model = GradientBoostingRegressor(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_valid)

    rmse = np.sqrt(mean_squared_error(y_valid, y_pred))

    mape = mean_absolute_percentage_error(y_valid, y_pred)
    trial.set_user_attr("mape", mape)
    
    return rmse

In [10]:
%%time

result_best_gbr_rmse = []
result_best_gbr_mape = []

for i, n in all_data.items():

    line_length = 150
    print('-' * line_length)
    print(f'Лучшие параметры для датасета (модель GradientBoostingRegressor): {i}'.center(line_length))
    print('-' * line_length)

    optuna.logging.set_verbosity(optuna.logging.ERROR)
    study = optuna.create_study(direction="minimize")
    study.optimize(partial(objective_gbr, data=n), n_trials=10)

    print("Лучшие параметры:", study.best_params)
    print("Лучшее RMSE:", study.best_value)

    result_best_gbr_rmse.append(study.best_value)

    df = study.trials_dataframe()
    best_row = df.loc[df["value"].idxmin()]
    best_mape = round(best_row["user_attrs_mape"] * 100, 4)
    result_best_gbr_mape.append(best_mape)
    

all_reault['GBR_result_rmse'] = result_best_gbr_rmse
all_reault['GBR_result_mape'] = result_best_gbr_mape

------------------------------------------------------------------------------------------------------------------------------------------------------
                                   Лучшие параметры для датасета (модель GradientBoostingRegressor): crypt_Ethereum                                   
------------------------------------------------------------------------------------------------------------------------------------------------------
Лучшие параметры: {'n_estimators': 95, 'learning_rate': 0.19499022636329993, 'max_depth': 4, 'subsample': 0.9085858571643189, 'min_samples_split': 9, 'min_samples_leaf': 3}
Лучшее RMSE: 131.73817983907912
------------------------------------------------------------------------------------------------------------------------------------------------------
                                     Лучшие параметры для датасета (модель GradientBoostingRegressor): crypt_BTC                                      
----------------------------------------

In [19]:
def objective_lgbm(trial, data):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "num_leaves": trial.suggest_int("num_leaves", 7, 255),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 30)
    }

    train_raw, valid_raw, _ = split_data(data)
    tail_len = 30

    train = upgrade_dataset(train_raw)
    valid_with_tail = pd.concat([train_raw.tail(tail_len), valid_raw])
    valid_with_tail = upgrade_dataset(valid_with_tail)
    valid = valid_with_tail.loc[valid_raw.index]

    X_train, y_train = return_x_y(train)
    X_valid, y_valid = return_x_y(valid)

    model = LGBMRegressor(**params, verbose = -1)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_valid)

    rmse = np.sqrt(mean_squared_error(y_valid, y_pred))

    mape = mean_absolute_percentage_error(y_valid, y_pred)
    trial.set_user_attr("mape", mape)
    
    return rmse

In [20]:
%%time

result_best_lgbm_rmse = []
result_best_lgbm_mape = []

for i, n in all_data.items():

    line_length = 150
    print('-' * line_length)
    print(f'Лучшие параметры для датасета (модель LGBMRegressor): {i}'.center(line_length))
    print('-' * line_length)

    optuna.logging.set_verbosity(optuna.logging.ERROR)
    study = optuna.create_study(direction="minimize")
    study.optimize(partial(objective_lgbm, data=n), n_trials=10)

    print("Лучшие параметры:", study.best_params)
    print("Лучшее RMSE:", study.best_value)

    result_best_lgbm_rmse.append(study.best_value)

    df = study.trials_dataframe()
    best_row = df.loc[df["value"].idxmin()]
    best_mape = round(best_row["user_attrs_mape"] * 100, 4)
    result_best_lgbm_mape.append(best_mape)
    

all_reault['LGBM_result_rmse'] = result_best_lgbm_rmse
all_reault['LGBM_result_mape'] = result_best_lgbm_mape

------------------------------------------------------------------------------------------------------------------------------------------------------
                                         Лучшие параметры для датасета (модель LGBMRegressor): crypt_Ethereum                                         
------------------------------------------------------------------------------------------------------------------------------------------------------
Лучшие параметры: {'n_estimators': 230, 'learning_rate': 0.027678543029969038, 'max_depth': 4, 'num_leaves': 239, 'min_child_samples': 16}
Лучшее RMSE: 122.55317576671406
------------------------------------------------------------------------------------------------------------------------------------------------------
                                           Лучшие параметры для датасета (модель LGBMRegressor): crypt_BTC                                            
--------------------------------------------------------------------------

In [21]:
class TunedLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(1)

In [22]:
class SlidingWindowDataset(Dataset):
    def __init__(self, X, y, seq_len=10):
        self.X_seq = []
        self.y = []

        for i in range(len(X) - seq_len):
            self.X_seq.append(X.iloc[i:i+seq_len].values)
            self.y.append(y.iloc[i+seq_len])

        self.X_seq = torch.tensor(self.X_seq, dtype=torch.float32)
        self.y = torch.tensor(self.y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_seq[idx], self.y[idx]


In [23]:
class DeepLSTMRegressor(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                            num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):  # x: [batch, seq_len, features]
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])  # последний шаг
        return out.squeeze(1)


In [27]:
def objective_lstm(trial, data):
    # Гиперпараметры
    hidden_size = trial.suggest_int("hidden_size", 32, 256)
    num_layers = trial.suggest_int("num_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    seq_len = trial.suggest_int("seq_len", 5, 30)

    # Масштабируем данные
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()
    
    train_raw, valid_raw, _ = split_data(data)
    tail_len = 30
    train = upgrade_dataset(train_raw)
    valid_with_tail = pd.concat([train_raw.tail(tail_len), valid_raw])
    valid_with_tail = upgrade_dataset(valid_with_tail)
    valid = valid_with_tail.loc[valid_raw.index]
    
    X_train, y_train = return_x_y(train)
    X_valid, y_valid = return_x_y(valid)
    
    
    X_train_scaled = pd.DataFrame(scaler_X.fit_transform(X_train), columns=X_train.columns)
    X_valid_scaled = pd.DataFrame(scaler_X.transform(X_valid), columns=X_valid.columns)
    
    y_train_scaled = pd.Series(scaler_y.fit_transform(y_train.values.reshape(-1, 1)).flatten())
    y_valid_scaled = pd.Series(scaler_y.transform(y_valid.values.reshape(-1, 1)).flatten())

    
    # Обновляем датасет
    train_ds = SlidingWindowDataset(X_train_scaled, y_train_scaled, seq_len)
    val_ds = SlidingWindowDataset(X_valid_scaled, y_valid_scaled, seq_len)

    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=32)

    model = TunedLSTM(input_size=X_train.shape[1], hidden_size=hidden_size,
                      num_layers=num_layers, dropout=dropout)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()
    for epoch in range(10):  # не слишком долго
        for x_batch, y_batch in train_loader:
            output = model(x_batch)
            loss = criterion(output, y_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Предсказание
    model.eval()
    y_preds = []
    with torch.no_grad():
        for x_batch, _ in val_loader:
            pred = model(x_batch)
            y_preds.extend(pred.numpy())

    # обратная нормализация
    y_preds_rescaled = scaler_y.inverse_transform(np.array(y_preds).reshape(-1, 1)).flatten()
    y_valid_rescaled = scaler_y.inverse_transform(
        y_valid_scaled.values[seq_len:].reshape(-1, 1)
    ).flatten()

    rmse = np.sqrt(mean_squared_error(y_valid_rescaled, y_preds_rescaled))

    mape = mean_absolute_percentage_error(y_valid_rescaled, y_preds_rescaled)
    trial.set_user_attr("mape", mape)
    
    return rmse


In [28]:
%%time

result_best_lstm_rmse = []
result_best_lstm_mape = []

for i, n in all_data.items():

    line_length = 150
    print('-' * line_length)
    print(f'Лучшие параметры для датасета (нейросеть LSTM) {i}'.center(line_length))
    print('-' * line_length)

    optuna.logging.set_verbosity(optuna.logging.ERROR)
    study = optuna.create_study(direction="minimize")
    study.optimize(partial(objective_lstm, data=n), n_trials=10)

    print("Лучшие параметры:", study.best_params)
    print("Лучшее RMSE:", study.best_value)

    result_best_lstm_rmse.append(study.best_value)

    df = study.trials_dataframe()
    best_row = df.loc[df["value"].idxmin()]
    best_mape = round(best_row["user_attrs_mape"] * 100, 4)
    result_best_lstm_mape.append(best_mape)
    

all_reault['LSTM_result_rmse'] = result_best_lstm_rmse
all_reault['LSTM_result_mape'] = result_best_lstm_mape

------------------------------------------------------------------------------------------------------------------------------------------------------
                                            Лучшие параметры для датасета (нейросеть LSTM) crypt_Ethereum                                             
------------------------------------------------------------------------------------------------------------------------------------------------------


/usr/local/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2827197557290514 and num_layers=1
  warnings.warn(
/usr/local/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.4693569626639834 and num_layers=1
  warnings.warn(
/usr/local/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.18829113379578205 and num_layers=1
  warnings.warn(
/usr/local/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers grea

Лучшие параметры: {'hidden_size': 194, 'num_layers': 1, 'dropout': 0.4693569626639834, 'lr': 0.0009108165339944688, 'seq_len': 7}
Лучшее RMSE: 151.17188328188763
------------------------------------------------------------------------------------------------------------------------------------------------------
                                               Лучшие параметры для датасета (нейросеть LSTM) crypt_BTC                                               
------------------------------------------------------------------------------------------------------------------------------------------------------


/usr/local/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.19124522214859976 and num_layers=1
  warnings.warn(


Лучшие параметры: {'hidden_size': 123, 'num_layers': 3, 'dropout': 0.4786437413940502, 'lr': 0.00011574221865717214, 'seq_len': 6}
Лучшее RMSE: 3273.7512863772617
------------------------------------------------------------------------------------------------------------------------------------------------------
                                          Лучшие параметры для датасета (нейросеть LSTM) futures_Brent_LCOK5                                          
------------------------------------------------------------------------------------------------------------------------------------------------------


...

Лучшие параметры: {'hidden_size': 232, 'num_layers': 1, 'dropout': 0.2561411777470626, 'lr': 0.0003220194675159127, 'seq_len': 29}
Лучшее RMSE: 8.077635647818948
------------------------------------------------------------------------------------------------------------------------------------------------------
                                        Лучшие параметры для датасета (нейросеть LSTM) spot_preciouse_metal_AG                                        
------------------------------------------------------------------------------------------------------------------------------------------------------


/usr/local/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.22221912191659032 and num_layers=1
  warnings.warn(
/usr/local/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.22727595420729668 and num_layers=1
  warnings.warn(
/usr/local/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2908311773478998 and num_layers=1
  warnings.warn(


Лучшие параметры: {'hidden_size': 219, 'num_layers': 1, 'dropout': 0.2908311773478998, 'lr': 0.0015684231181917073, 'seq_len': 6}
Лучшее RMSE: 1.2015274162661123
------------------------------------------------------------------------------------------------------------------------------------------------------
                                        Лучшие параметры для датасета (нейросеть LSTM) spot_preciouse_metal_AU                                        
------------------------------------------------------------------------------------------------------------------------------------------------------


/usr/local/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.20335858526667666 and num_layers=1
  warnings.warn(
/usr/local/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.17053855163576737 and num_layers=1
  warnings.warn(
/usr/local/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.4541470129942794 and num_layers=1
  warnings.warn(
/usr/local/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers gre

Лучшие параметры: {'hidden_size': 157, 'num_layers': 1, 'dropout': 0.17053855163576737, 'lr': 0.0001086892714804816, 'seq_len': 16}
Лучшее RMSE: 192.31999818153335
CPU times: user 56min 23s, sys: 1min 45s, total: 58min 8s
Wall time: 16min 48s


In [29]:
all_reault

,dataset_name,dataset_mean_price,RF_result_rmse,RF_result_mape,GBR_result_rmse,GBR_result_mape,LGBM_result_rmse,LGBM_result_mape,LSTM_result_rmse,LSTM_result_mape
0,crypt_Ethereum,1264.224518,125.919678,3.0891,131.738180,3.3340,122.553176,3.0723,151.171883,3.7454
1,crypt_BTC,24007.290015,3569.341199,4.5163,2945.854821,3.9519,3181.906238,4.0354,3273.751286,4.4539
2,futures_Brent_LCOK5,68.036667,1.771514,1.6869,1.717633,1.6320,1.699224,1.6636,1.969132,1.9323
3,futures_WTI_CLJ5,63.524965,1.641177,1.7065,1.536987,1.6256,1.521592,1.5939,1.854264,1.9764
4,share_metal_jiangxi_copper,10.239784,0.461733,2.5564,0.435104,2.4063,0.471184,2.5368,0.562137,3.2470
5,share_metal_baoshan_iron,5.255690,0.145714,1.6661,0.141912,1.6503,0.142493,1.6546,0.170736,2.0387
6,share_oil_petrochina_hk,3.622050,1.266178,15.0481,1.228852,14.3215,1.262400,14.8207,1.088721,13.4905
7,share_oil_equinor_oslo,187.087574,8.015208,2.3777,7.089783,1.9602,6.867711,1.9750,8.077636,2.4248
8,spot_preciouse_metal_AG,20.814301,1.765372,4.2537,1.647971,3.9673,1.601554,3.8543,1.201527,3.1623
9,spot_preciouse_metal_AU,1677.948521,301.369130,9.8997,301.095532,9.8270,298.067235,9.7684,192.319998,6.4860
